# Notebook 01 — Data Audit, Leakage Control, Group Split and Manifests (Phase B)

**Project:** Bahnar → Vietnamese Speech Translation  
**RQ1:** Controlled comparison between Cascaded ASR+MT and Direct S2TT

## Phase B goals

1. Treat HF `speaker_id` as **`source_label`** (program/channel), not a person.
2. Build **`recording_group_id`** from record/audio prefixes (`1CO_01_001` → `1CO_01`).
3. Decode/QA **only** accessible validation+test audio (~215). Train waveform QA is deferred to Notebook 02 data-loader/preflight.
4. Freeze `rq1_test` from QA hard-pass records (quality warnings do not auto-drop).
5. Group-split train/validation by `group_id` (multi-candidate, 8–12% val band).
6. Export reproducible manifests — **no model training** in this notebook.

`GROUP_REVIEW_APPROVED` stays `False` until group diagnostics look reasonable.


## Mapping với kế hoạch RQ1

| Phần notebook | Step trong phân tích | Output |
|---|---|---|
| Setup + schema | Step 1 | Dataset revision, schema |
| Metadata audit | Step 2 | Audit reports, eligible rows |
| Normalize + group | Step 3 và 5 | Duplicate keys, `group_id_review.csv` |
| Evaluation + leakage | Step 4 | Locked evaluation candidates, excluded leakage |
| Group split | Step 4 | Pseudo-train và pseudo-validation |
| Export | Step 6 | `rq1_train`, `rq1_validation`, `rq1_test` |

Notebook 01 **không train model**. Fine-tuning bắt đầu từ Notebook 02.

In [8]:
# Cell 1 — Install dependencies (IDE + Colab bootstrap)
# Prefer kernel: Python (bahnar-s2tt). Do not hardcode personal machine paths.
from pathlib import Path
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

def resolve_project_root(*, drive_mounted: bool = False) -> Path:
    """Find bahnar-s2tt-thesis root from cwd/parents, or Colab paths after Drive mount."""
    if IN_COLAB and drive_mounted:
        for cand in (
            Path("/content/drive/MyDrive/bahnar-s2tt-thesis"),
            Path("/content/bahnar-s2tt-thesis"),
        ):
            if (cand / "requirements.txt").is_file() and (cand / "src").is_dir():
                return cand.resolve()

    cwd = Path.cwd().resolve()
    for cand in [cwd, cwd.parent, *cwd.parents]:
        if (cand / "requirements.txt").is_file() and (cand / "src").is_dir():
            return cand
        if cand.name == "bahnar-s2tt-thesis" and (cand / "requirements.txt").is_file():
            return cand
    raise FileNotFoundError(
        "Cannot locate project root (needs requirements.txt + src/). "
        "Open the notebook from bahnar-s2tt-thesis/notebooks/ or the project root, "
        "or on Colab mount Drive and place the repo at MyDrive/bahnar-s2tt-thesis."
    )

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = resolve_project_root(drive_mounted=True)
else:
    PROJECT_ROOT = resolve_project_root(drive_mounted=False)

REQ = PROJECT_ROOT / "requirements.txt"
if not REQ.is_file():
    raise FileNotFoundError(f"requirements.txt not found at {REQ}")
if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError(f"src/ not found under {PROJECT_ROOT}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Installing from:", REQ)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(REQ)])
print("Dependencies installed.")


PROJECT_ROOT: /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis
Installing from: /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/requirements.txt
Dependencies installed.


In [9]:
# Cell 2 — Imports, reproducibility and project directories
import hashlib
import json
import os
import random
import re
import subprocess
import sys
import unicodedata
from pathlib import Path
from urllib.parse import unquote, urlparse

import duckdb
import numpy as np
import pandas as pd
import requests
from huggingface_hub import HfApi
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

IN_COLAB = "google.colab" in sys.modules

def _project_root_valid(path) -> bool:
    try:
        root = Path(path)
        return root.is_dir() and (root / "requirements.txt").is_file() and (root / "src").is_dir()
    except Exception:
        return False

def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    search = [cwd, cwd.parent, *cwd.parents]
    if IN_COLAB:
        search = [
            Path("/content/drive/MyDrive/bahnar-s2tt-thesis"),
            Path("/content/bahnar-s2tt-thesis"),
            *search,
        ]
    for cand in search:
        if (cand / "requirements.txt").is_file() and (cand / "src").is_dir():
            return cand.resolve()
    raise FileNotFoundError(
        "Cannot locate project root (needs requirements.txt + src/). "
        "Run Cell 1 first (Colab mounts Drive there)."
    )

# Reuse PROJECT_ROOT from Cell 1 when valid; do NOT remount Drive here.
if "PROJECT_ROOT" in globals() and _project_root_valid(PROJECT_ROOT):
    PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
else:
    PROJECT_ROOT = resolve_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.split_utils import (
    collect_ambiguous_group_suffixes,
    derive_group,
    namespace_cross_source_collisions,
    select_best_group_split,
    stratified_group_review_sample,
)
from src.audio_utils import check_waveform

DIRS = {
    "audit": PROJECT_ROOT / "data" / "audit",
    "manifests": PROJECT_ROOT / "data" / "manifests",
    "configs": PROJECT_ROOT / "configs",
    "checkpoints": PROJECT_ROOT / "checkpoints",
    "predictions": PROJECT_ROOT / "predictions",
    "metrics": PROJECT_ROOT / "metrics",
    "results": PROJECT_ROOT / "results",
}
for folder in DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Outputs will persist under:", DIRS["manifests"])


PROJECT_ROOT: /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis
Outputs will persist under: /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/data/manifests


## Configuration

- `GROUP_REVIEW_APPROVED=False` on first Phase B run: inspect `recording_group_id` diagnostics first.
- Train waveform QA is **not** run here (deferred to Notebook 02).
- `REMOVE_EXACT_TEXT_PAIR_OVERLAP=True`: conservative train↔test text-pair decontamination.
- Dataset Hub commit SHA must match `EXPECTED_DATASET_REVISION` (locked; not auto-updated).


In [10]:
# Cell 3 — Experiment configuration
DATASET_ID = "cuong06/Bahnar_Vietnamese"
CONFIG_NAME = "default"

EXPECTED_DATASET_REVISION = "3d88d3951b1a6e3388559b341cd7bd274879d696"

VALIDATION_FRACTION = 0.10  # target ~8–12% by groups
VALIDATION_FRACTION_MIN = 0.08
VALIDATION_FRACTION_MAX = 0.12
SPLIT_N_CANDIDATES = 200
DROP_EXACT_DUPLICATE_ROWS = True
REMOVE_EXACT_TEXT_PAIR_OVERLAP = True

# IMPORTANT: keep False until recording_group_id review looks good.
GROUP_REVIEW_APPROVED = False

LARGE_GROUP_WARNING_FRACTION = 0.20
AUDIO_MIN_DURATION = 0.05
AUDIO_MAX_DURATION = 120.0
EXPECTED_FROZEN_TEST_ROWS = 215

EXPECTED_ROW_COUNTS = {
    "train": 113_830,
    "validation": 2_636,
    "test": 3_336,
}
EXPECTED_ACCESSIBLE_AUDIO = {
    "validation": 205,
    "test": 10,
}

print(json.dumps({
    "dataset": DATASET_ID,
    "expected_revision": EXPECTED_DATASET_REVISION,
    "validation_fraction": VALIDATION_FRACTION,
    "validation_band": [VALIDATION_FRACTION_MIN, VALIDATION_FRACTION_MAX],
    "split_n_candidates": SPLIT_N_CANDIDATES,
    "group_review_approved": GROUP_REVIEW_APPROVED,
    "train_waveform_qa": "deferred_to_notebook_02",
}, indent=2))


{
  "dataset": "cuong06/Bahnar_Vietnamese",
  "expected_revision": "3d88d3951b1a6e3388559b341cd7bd274879d696",
  "validation_fraction": 0.1,
  "validation_band": [
    0.08,
    0.12
  ],
  "split_n_candidates": 200,
  "group_review_approved": false,
  "train_waveform_qa": "deferred_to_notebook_02"
}


In [11]:
# Cell 4 — Optional Hugging Face authentication
# A free HF token avoids anonymous rate-limit warnings. Never print the token.
HF_TOKEN = os.environ.get("HF_TOKEN")
if IN_COLAB and not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

HTTP_HEADERS = {"User-Agent": "bahnar-s2tt-thesis-audit/1.0"}
if HF_TOKEN:
    HTTP_HEADERS["Authorization"] = f"Bearer {HF_TOKEN}"
    print("HF authentication: enabled")
else:
    print("HF authentication: anonymous (works for this public dataset)")

HF authentication: anonymous (works for this public dataset)


## Step 1 — Resolve dataset files and schema

Dataset Server trả về danh sách converted Parquet. DuckDB chỉ chọn các cột metadata cần thiết như text, ID và `audio.path`; nó **không chủ động decode audio bytes**. Đây là lý do không dùng `load_dataset()` để quét toàn bộ 654 giờ ngay trong notebook này.

In [12]:
# Cell 5 — Lock Hub revision + Dataset Server helpers / public row counts
DATASET_SERVER = "https://datasets-server.huggingface.co"

def dataset_server_get(endpoint, **params):
    response = requests.get(
        f"{DATASET_SERVER}/{endpoint}",
        params=params,
        headers=HTTP_HEADERS,
        timeout=120,
    )
    response.raise_for_status()
    return response.json()

def fetch_hub_dataset_sha(dataset_id: str) -> str:
    """Authoritative dataset commit SHA from Hugging Face Hub (not Dataset Server)."""
    api = HfApi()
    info = api.dataset_info(dataset_id, files_metadata=False)
    sha = getattr(info, "sha", None) or getattr(info, "revision", None)
    if not sha:
        raise RuntimeError(
            f"Hub dataset_info({dataset_id}) did not return a commit SHA. "
            "Cannot lock dataset revision."
        )
    return str(sha)

DATASET_REVISION_BEFORE = fetch_hub_dataset_sha(DATASET_ID)
print("Hub dataset SHA (before audit):", DATASET_REVISION_BEFORE)
if DATASET_REVISION_BEFORE != EXPECTED_DATASET_REVISION:
    raise RuntimeError(
        "Dataset revision mismatch.\n"
        f"  expected: {EXPECTED_DATASET_REVISION}\n"
        f"  actual:   {DATASET_REVISION_BEFORE}\n"
        "Refusing to continue. Do not auto-update EXPECTED_DATASET_REVISION; "
        "investigate upstream changes first."
    )
print("Revision lock OK (matches EXPECTED_DATASET_REVISION).")

size_payload = dataset_server_get("size", dataset=DATASET_ID)
parquet_payload = dataset_server_get("parquet", dataset=DATASET_ID)

public_counts = {
    item["split"]: int(item["num_rows"])
    for item in size_payload["size"]["splits"]
    if item.get("config") == CONFIG_NAME
}

parquet_rows = [
    item for item in parquet_payload.get("parquet_files", [])
    if item.get("config") == CONFIG_NAME
]
parquet_by_split = {
    split: [x["url"] for x in parquet_rows if x.get("split") == split]
    for split in ("train", "validation", "test")
}

print("Public row counts:")
for split in ("train", "validation", "test"):
    print(f"  {split:10s}: {public_counts.get(split, 0):,} rows | "
          f"{len(parquet_by_split.get(split, []))} parquet shards")

# Informational only — empty/missing Dataset Server revision is NOT a lock.
server_revisions = {
    key: parquet_payload.get(key)
    for key in ("dataset_git_revision", "parquet_revision")
    if parquet_payload.get(key) is not None
}
print("Dataset Server revisions (informational, not authoritative):", server_revisions or "{}")

for split, expected in EXPECTED_ROW_COUNTS.items():
    actual = public_counts.get(split)
    if actual != expected:
        print(f"WARNING: {split} expected {expected:,}, current API reports {actual:,}")


Hub dataset SHA (before audit): 3d88d3951b1a6e3388559b341cd7bd274879d696
Revision lock OK (matches EXPECTED_DATASET_REVISION).
Public row counts:
  train     : 113,830 rows | 50 parquet shards
  validation: 2,636 rows | 1 parquet shards
  test      : 3,336 rows | 1 parquet shards
Dataset Server revisions (informational, not authoritative): {}


In [13]:
# Cell 6 — DuckDB remote-Parquet metadata scanner
con = duckdb.connect()
try:
    con.execute("INSTALL httpfs")
except Exception:
    pass
con.execute("LOAD httpfs")

def sql_string(value):
    return "'" + str(value).replace("'", "''") + "'"

def sql_identifier(value):
    return '"' + str(value).replace('"', '""') + '"'

def parquet_source(urls):
    if not urls:
        raise ValueError("No Parquet URLs were returned for this split")
    url_list = "[" + ",".join(sql_string(u) for u in urls) + "]"
    return (
        f"read_parquet({url_list}, union_by_name=true, "
        "filename=true, file_row_number=true)"
    )

def split_schema(split):
    source = parquet_source(parquet_by_split[split])
    return con.execute(f"DESCRIBE SELECT * FROM {source}").fetchdf()

schemas = {split: split_schema(split) for split in ("train", "validation", "test")}
display(schemas["train"])

,column_name,column_type,null,key,default,extra
0,id,VARCHAR,YES,None,None,None
1,speaker_id,VARCHAR,YES,None,None,None
2,audio,"STRUCT(bytes BLOB, path VARCHAR)",YES,None,None,None
3,duration,FLOAT,YES,None,None,None
4,text_vi,VARCHAR,YES,None,None,None
5,text_bahnar,VARCHAR,YES,None,None,None
6,text_en,VARCHAR,YES,None,None,None
7,file_row_number,BIGINT,YES,None,None,None
8,filename,VARCHAR,YES,None,None,None


In [14]:
# Cell 7 — Select canonical metadata without loading audio waveforms
ID_CANDIDATES = ["id", "audio_id", "utterance_id", "uid", "key"]
BAHNAR_TEXT_CANDIDATES = [
    "text_bahnar", "bahnar_text", "sentence_bahnar", "transcript_bahnar", "transcript"
]
VI_TEXT_CANDIDATES = [
    "text_vi", "vietnamese_text", "sentence_vi", "translation_vi", "translation"
]
EN_TEXT_CANDIDATES = ["text_en", "english_text", "sentence_en"]
SPEAKER_CANDIDATES = ["speaker_id", "speaker", "speaker_name"]
DURATION_CANDIDATES = ["duration", "duration_seconds", "audio_duration"]
EXPLICIT_GROUP_COLUMNS = [
    "video_id", "source_video_id", "youtube_id", "recording_id", "session_id",
    "source_id", "source", "video_url", "url", "playlist_id", "channel_id"
]

def first_present(columns, candidates):
    return next((c for c in candidates if c in columns), None)

def canonical_expr(columns, candidates, alias):
    present = [c for c in candidates if c in columns]
    if not present:
        return f"CAST(NULL AS VARCHAR) AS {sql_identifier(alias)}"
    values = [f"CAST({sql_identifier(c)} AS VARCHAR)" for c in present]
    return f"COALESCE({', '.join(values)}) AS {sql_identifier(alias)}"

def scan_split_metadata(split):
    schema = schemas[split]
    columns = set(schema["column_name"].tolist())
    source = parquet_source(parquet_by_split[split])

    select_exprs = [
        f"{sql_string(split)} AS source_split",
        "filename AS parquet_file",
        "file_row_number AS shard_row_index",
        canonical_expr(columns, ID_CANDIDATES, "record_id"),
        canonical_expr(columns, BAHNAR_TEXT_CANDIDATES, "text_bahnar"),
        canonical_expr(columns, VI_TEXT_CANDIDATES, "text_vi"),
        canonical_expr(columns, EN_TEXT_CANDIDATES, "text_en"),
        canonical_expr(columns, SPEAKER_CANDIDATES, "speaker_id"),  # raw HF field
    ]

    duration_col = first_present(columns, DURATION_CANDIDATES)
    if duration_col:
        select_exprs.append(
            f"TRY_CAST({sql_identifier(duration_col)} AS DOUBLE) AS duration_seconds"
        )
    else:
        select_exprs.append("CAST(NULL AS DOUBLE) AS duration_seconds")

    audio_type = ""
    if "audio" in columns:
        match = schema.loc[schema["column_name"] == "audio", "column_type"]
        audio_type = str(match.iloc[0]) if len(match) else ""

    if "audio" in columns and "STRUCT" in audio_type.upper():
        select_exprs.extend([
            "CAST(struct_extract(\"audio\", 'path') AS VARCHAR) AS audio_path",
            "(struct_extract(\"audio\", 'path') IS NOT NULL) AS audio_available",
        ])
    elif "audio" in columns:
        select_exprs.extend([
            "CAST(\"audio\" AS VARCHAR) AS audio_path",
            "(\"audio\" IS NOT NULL) AS audio_available",
        ])
    elif "path" in columns:
        select_exprs.extend([
            "CAST(\"path\" AS VARCHAR) AS audio_path",
            "(\"path\" IS NOT NULL) AS audio_available",
        ])
    else:
        select_exprs.extend([
            "CAST(NULL AS VARCHAR) AS audio_path",
            "FALSE AS audio_available",
        ])

    for column in EXPLICIT_GROUP_COLUMNS:
        if column in columns:
            select_exprs.append(
                f"CAST({sql_identifier(column)} AS VARCHAR) AS {sql_identifier(column)}"
            )

    query = f"SELECT {', '.join(select_exprs)} FROM {source}"
    return con.execute(query).fetchdf()

metadata = {}
for split in ("train", "validation", "test"):
    print(f"Scanning {split} metadata...")
    df = scan_split_metadata(split)
    # Owner-confirmed: HF speaker_id is a program/channel label, not a person.
    df["source_label"] = df["speaker_id"]
    metadata[split] = df
    print(f"  scanned: {len(metadata[split]):,}")
    print(f"  unique source_label: {metadata[split]['source_label'].nunique(dropna=True):,}")

# Re-fetch Hub SHA after metadata scan; must match pre-audit lock.
DATASET_REVISION_AFTER = fetch_hub_dataset_sha(DATASET_ID)
print("Hub dataset SHA (after metadata scan):", DATASET_REVISION_AFTER)
if DATASET_REVISION_AFTER != DATASET_REVISION_BEFORE:
    raise RuntimeError(
        "Dataset revision changed during metadata scan.\n"
        f"  before: {DATASET_REVISION_BEFORE}\n"
        f"  after:  {DATASET_REVISION_AFTER}"
    )
if DATASET_REVISION_AFTER != EXPECTED_DATASET_REVISION:
    raise RuntimeError(
        "Post-scan Hub SHA does not match EXPECTED_DATASET_REVISION.\n"
        f"  expected: {EXPECTED_DATASET_REVISION}\n"
        f"  actual:   {DATASET_REVISION_AFTER}"
    )
print("Post-scan revision lock OK.")


Scanning train metadata...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  scanned: 113,830
  unique source_label: 37
Scanning validation metadata...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  scanned: 2,636
  unique source_label: 4
Scanning test metadata...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  scanned: 3,336
  unique source_label: 4
Hub dataset SHA (after metadata scan): 3d88d3951b1a6e3388559b341cd7bd274879d696
Post-scan revision lock OK.


> **Audio availability note:** metadata `audio_available` is inferred from `audio.path` (no waveform decode).  
> **Source label note:** column `source_label` copies HF `speaker_id` (e.g. `oneway`, `yt_53`) — treated as **source/program**, not a speaker identity.  
> Decode QA for accessible validation+test (~215) runs in Step 3b. Train waveform QA is deferred to Notebook 02.


## Step 2 — Full metadata audit

A row is temporarily RQ1-eligible when:

```text
audio reference available
AND text_bahnar not empty
AND text_vi not empty
```

This is **metadata eligibility** only. Waveform QA happens after grouping.


In [15]:
# Cell 8 — Text presence, eligibility and audit summary
def has_text(series):
    return series.fillna("").astype(str).str.strip().ne("")

def add_availability_flags(df):
    out = df.copy()
    out["bahnar_text_available"] = has_text(out["text_bahnar"])
    out["vietnamese_text_available"] = has_text(out["text_vi"])
    out["english_text_available"] = has_text(out["text_en"])
    out["audio_available"] = out["audio_available"].fillna(False).astype(bool)
    out["eligible_rq1_metadata"] = (
        out["audio_available"]
        & out["bahnar_text_available"]
        & out["vietnamese_text_available"]
    )
    return out

metadata = {k: add_availability_flags(v) for k, v in metadata.items()}

audit_rows = []
for split, df in metadata.items():
    audit_rows.append({
        "split": split,
        "total_rows": len(df),
        "audio_reference_available": int(df["audio_available"].sum()),
        "audio_percent": round(100 * df["audio_available"].mean(), 2),
        "bahnar_text_available": int(df["bahnar_text_available"].sum()),
        "vietnamese_text_available": int(df["vietnamese_text_available"].sum()),
        "english_text_available": int(df["english_text_available"].sum()),
        "eligible_rq1_metadata": int(df["eligible_rq1_metadata"].sum()),
        "unique_record_ids": int(df["record_id"].nunique(dropna=True)),
        "unique_source_labels": int(df["source_label"].nunique(dropna=True)),
    })

audit_summary = pd.DataFrame(audit_rows)
display(audit_summary)
audit_summary.to_csv(DIRS["audit"] / "metadata_audit_summary.csv", index=False)

for split, expected in EXPECTED_ACCESSIBLE_AUDIO.items():
    actual = int(metadata[split]["audio_available"].sum())
    if actual != expected:
        print(
            f"REVIEW REQUIRED: {split} audio-path proxy={actual:,}; "
            f"previous direct audit={expected:,}."
        )


,split,total_rows,audio_reference_available,audio_percent,bahnar_text_available,vietnamese_text_available,english_text_available,eligible_rq1_metadata,unique_record_ids,unique_source_labels
0,train,113830,113830,100.00,113830,113830,113830,113830,113830,37
1,validation,2636,205,7.78,2636,2636,2441,205,2636,4
2,test,3336,10,0.30,3336,3336,3336,10,3336,4


In [16]:
# Cell 9 — Duplicate and source_label diagnostics
duplicate_reports = []

for split, df in metadata.items():
    duplicate_id_count = int(
        df.loc[has_text(df["record_id"]), "record_id"].duplicated(keep=False).sum()
    )
    duplicate_pair_count = int(
        df[["text_bahnar", "text_vi"]]
        .fillna("")
        .astype(str)
        .duplicated(keep=False)
        .sum()
    )
    duplicate_reports.append({
        "split": split,
        "rows_in_duplicate_id_groups": duplicate_id_count,
        "rows_in_duplicate_raw_text_pair_groups": duplicate_pair_count,
    })

    source_counts = (
        df.loc[df["audio_available"], "source_label"]
        .fillna("<missing>")
        .value_counts()
        .rename_axis("source_label")
        .reset_index(name="rows")
    )
    source_counts.to_csv(DIRS["audit"] / f"{split}_audio_by_source_label.csv", index=False)

duplicate_summary = pd.DataFrame(duplicate_reports)
display(duplicate_summary)
duplicate_summary.to_csv(DIRS["audit"] / "duplicate_summary_before_normalization.csv", index=False)

print("Validation audio grouped by source_label (program/channel):")
display(pd.read_csv(DIRS["audit"] / "validation_audio_by_source_label.csv").head(20))

# Peek ID patterns to support recording_group_id design
print("\nSample record_id / audio_path / source_label (train eligible head):")
peek = metadata["train"].loc[metadata["train"]["eligible_rq1_metadata"],
                             ["record_id", "audio_path", "source_label", "duration_seconds"]].head(30)
display(peek)
peek.to_csv(DIRS["audit"] / "train_id_pattern_peek.csv", index=False)


,split,rows_in_duplicate_id_groups,rows_in_duplicate_raw_text_pair_groups
0,train,0,70
1,validation,0,125
2,test,0,78


Validation audio grouped by source_label (program/channel):


,source_label,rows
0,oneway,195
1,KT_PT,4
2,GL_YMS,3
3,BD_CTV_2,3



Sample record_id / audio_path / source_label (train eligible head):


,record_id,audio_path,source_label,duration_seconds
0,1CO_01_001,1CO_01_001.flac,KT_0,8.220000
1,1CO_01_002,1CO_01_002.flac,KT_0,17.260000
2,1CO_01_003,1CO_01_003.flac,KT_0,8.380000
3,1CO_01_004,1CO_01_004.flac,KT_0,6.920000
4,1CO_01_005,1CO_01_005.flac,KT_0,6.840000
5,1CO_01_006,1CO_01_006.flac,KT_0,6.350000
6,1CO_01_007,1CO_01_007.flac,KT_0,7.570000
7,1CO_01_008,1CO_01_008.flac,KT_0,7.980000
8,1CO_01_009,1CO_01_009.flac,KT_0,10.010000
9,1CO_01_010,1CO_01_010.flac,KT_0,13.270000


## Step 3 — Normalize keys and derive `recording_group_id`

Priority (see `src/split_utils.py`):

1. Explicit video/session columns (if present)
2. Prefix from `record_id` / `audio_path` (segment indices only when evidence supports it)
3. Fallback: `source_label`
4. Row-unique fallback

Numeric suffix rule: strip `_001` / `_10` / zero-padded `_0001`, but **do not** strip year-like `show_2025`.
Cross-source collisions on the same `recording_group_id` are namespaced into `group_id`.


In [17]:
# Cell 10 — Stable normalization and hash keys
WHITESPACE_RE = re.compile(r"\s+")

def normalize_key_text(value):
    if value is None or pd.isna(value):
        return ""
    text = unicodedata.normalize("NFC", str(value))
    text = WHITESPACE_RE.sub(" ", text).strip().casefold()
    return text

def sha1_text(value):
    return hashlib.sha1(value.encode("utf-8")).hexdigest()

def normalize_path_key(value):
    if value is None or pd.isna(value):
        return ""
    value = unquote(str(value)).replace("\\", "/").strip().casefold()
    return value

def add_stable_keys(df):
    out = df.copy()
    out["text_bahnar_key"] = out["text_bahnar"].map(normalize_key_text)
    out["text_vi_key"] = out["text_vi"].map(normalize_key_text)
    out["pair_key"] = (
        out["text_bahnar_key"] + "\x1f" + out["text_vi_key"]
    ).map(sha1_text)
    out["audio_key"] = out["audio_path"].map(normalize_path_key)
    out["record_uid"] = (
        out["source_split"].astype(str) + "|"
        + out["parquet_file"].astype(str) + "|"
        + out["shard_row_index"].astype(str)
    ).map(sha1_text)
    return out

metadata = {k: add_stable_keys(v) for k, v in metadata.items()}

In [18]:
# Cell 11 — Derive recording_group_id / group_id (+ collision namespace)
EXPLICIT_GROUP_COLUMNS = [
    "video_id", "source_video_id", "youtube_id", "recording_id", "session_id",
    "source_id", "source", "video_url", "url", "playlist_id", "channel_id",
]

for split in metadata:
    # Drop prior group cols if re-running
    for c in ("recording_group_id", "group_id", "group_source", "group_confidence", "group_suffix_note"):
        if c in metadata[split].columns:
            metadata[split] = metadata[split].drop(columns=[c])
    groups = metadata[split].apply(
        lambda row: derive_group(row, explicit_group_columns=EXPLICIT_GROUP_COLUMNS),
        axis=1,
    )
    metadata[split] = pd.concat([metadata[split], groups], axis=1)
    metadata[split], collisions = namespace_cross_source_collisions(metadata[split])
    collisions.to_csv(DIRS["audit"] / f"{split}_group_id_cross_source_collisions.csv", index=False)
    if len(collisions):
        print(f"{split}: namespaced {len(collisions)} cross-source recording_group_id collision(s)")

# Combined collision report
collision_frames = []
for split in ("train", "validation", "test"):
    path = DIRS["audit"] / f"{split}_group_id_cross_source_collisions.csv"
    if path.exists():
        tmp = pd.read_csv(path)
        if len(tmp):
            tmp.insert(0, "split", split)
            collision_frames.append(tmp)
cross_all = pd.concat(collision_frames, ignore_index=True) if collision_frames else pd.DataFrame(
    columns=["split", "recording_group_id", "n_source_labels", "source_labels", "rows"]
)
cross_all.to_csv(DIRS["audit"] / "group_id_cross_source_collisions.csv", index=False)

# Ambiguous suffixes (e.g. show_2025)
amb_frames = [collect_ambiguous_group_suffixes(metadata[s]) for s in metadata]
for i, s in enumerate(metadata):
    if len(amb_frames[i]):
        amb_frames[i] = amb_frames[i].copy()
        amb_frames[i].insert(0, "split", s)
ambiguous = pd.concat([a for a in amb_frames if len(a)], ignore_index=True) if any(len(a) for a in amb_frames) else pd.DataFrame()
if ambiguous.empty:
    ambiguous = pd.DataFrame(columns=["split", "record_id", "audio_path", "source_label", "group_id", "group_suffix_note"])
ambiguous.to_csv(DIRS["audit"] / "ambiguous_group_suffixes.csv", index=False)
print(f"Ambiguous group suffixes logged: {len(ambiguous):,}")

# Show whether prefix grouping works on known patterns
demo_ids = ["1CO_01_001", "1CO_01_002", "KT-XH02_002", "oneway_251021_1", "oneway_251021_10", "show_2025"]
print("Prefix demo:")
for rid in demo_ids:
    g = derive_group(pd.Series({
        "record_id": rid,
        "audio_path": f"{rid}.flac",
        "source_label": "demo",
        "record_uid": "x",
    }), explicit_group_columns=EXPLICIT_GROUP_COLUMNS)
    print(f"  {rid:20s} -> {g['group_id']} ({g['group_source']}, note={g.get('group_suffix_note')})")


train: namespaced 2285 cross-source recording_group_id collision(s)
validation: namespaced 23 cross-source recording_group_id collision(s)
test: namespaced 24 cross-source recording_group_id collision(s)
Ambiguous group suffixes logged: 0
Prefix demo:
  1CO_01_001           -> recording:1co_01 (record_id_prefix, note=None)
  1CO_01_002           -> recording:1co_01 (record_id_prefix, note=None)
  KT-XH02_002          -> recording:kt-xh02 (record_id_prefix, note=None)
  oneway_251021_1      -> recording:oneway_251021 (record_id_prefix, note=None)
  oneway_251021_10     -> recording:oneway_251021 (record_id_prefix, note=None)
  show_2025            -> source_label:demo (source_label, note=yearish_four_digit_suffix)


## Group Review Gate

Inspect stratified review CSVs before setting `GROUP_REVIEW_APPROVED=True`.

Checklist:

1. Most eligible rows should use `record_id_prefix` / `audio_path_prefix` (medium), not only `source_label`.
2. Largest `recording_group_id` fraction should not dominate (warn if > 20%).
3. Review sample covers **all** `source_label` values (not just `.head(500)`).
4. Check `ambiguous_group_suffixes.csv` and `group_id_cross_source_collisions.csv`.

If almost everything is still `source_label` / unresolved → **do not** approve yet.


In [19]:
# Cell 12 — Group diagnostics and stratified review files
group_review_frames = []

for split, df in metadata.items():
    eligible = df[df["eligible_rq1_metadata"]].copy()
    source_summary = (
        eligible.groupby(["group_source", "group_confidence"], dropna=False)
        .size()
        .reset_index(name="rows")
        .sort_values("rows", ascending=False)
    )
    source_summary.insert(0, "split", split)
    group_review_frames.append(source_summary)

    largest = (
        eligible.groupby(
            ["recording_group_id", "group_id", "group_source", "group_confidence"],
            dropna=False,
        )
        .size()
        .reset_index(name="rows")
        .sort_values("rows", ascending=False)
    )
    largest["fraction_of_eligible"] = largest["rows"] / max(len(eligible), 1)
    largest.to_csv(DIRS["audit"] / f"{split}_largest_groups.csv", index=False)

    review_cols = [
        c for c in [
            "record_uid", "record_id", "audio_path", "source_label", "duration_seconds",
            "recording_group_id", "group_id", "group_source", "group_confidence", "group_suffix_note",
        ] if c in eligible.columns
    ]
    examples = stratified_group_review_sample(
        eligible[review_cols],
        seed=SEED,
        max_per_source=20,
        max_per_group=2,
    )
    examples.to_csv(DIRS["audit"] / f"{split}_group_id_review.csv", index=False)
    print(
        f"{split}: review sample rows={len(examples):,} "
        f"sources={examples['source_label'].nunique(dropna=False) if len(examples) else 0}"
    )

group_source_summary = pd.concat(group_review_frames, ignore_index=True)
group_source_summary.to_csv(DIRS["audit"] / "group_source_summary.csv", index=False)
display(group_source_summary)

train_largest = pd.read_csv(DIRS["audit"] / "train_largest_groups.csv")
print("Largest train recording groups:")
display(train_largest.head(20))

if len(train_largest) and train_largest.iloc[0]["fraction_of_eligible"] > LARGE_GROUP_WARNING_FRACTION:
    print("WARNING: One train group occupies more than the configured threshold.")

elig = metadata["train"].loc[metadata["train"]["eligible_rq1_metadata"]]
n_medium = int((elig["group_confidence"] == "medium").sum())
n_low = int((elig["group_confidence"] == "low").sum())
n_unresolved = int((elig["group_confidence"] == "unresolved").sum())
print(f"Train eligible group_confidence: medium={n_medium:,} low={n_low:,} unresolved={n_unresolved:,}")
print("GROUP_REVIEW_APPROVED:", GROUP_REVIEW_APPROVED)
print("Wrote:", DIRS["audit"] / "group_source_summary.csv")
print("Wrote:", DIRS["audit"] / "train_largest_groups.csv")
print("Wrote:", DIRS["audit"] / "train_group_id_review.csv")
print("Wrote:", DIRS["audit"] / "validation_group_id_review.csv")
print("Wrote:", DIRS["audit"] / "test_group_id_review.csv")
print("Wrote:", DIRS["audit"] / "ambiguous_group_suffixes.csv")
print("Wrote:", DIRS["audit"] / "group_id_cross_source_collisions.csv")


train: review sample rows=601 sources=37
validation: review sample rows=30 sources=4
test: review sample rows=10 sources=2


,split,group_source,group_confidence,rows
0,train,record_id_prefix,medium,113830
1,validation,record_id_prefix,medium,205
2,test,record_id_prefix,medium,10


Largest train recording groups:


,recording_group_id,group_id,group_source,group_confidence,rows,fraction_of_eligible
0,98lc-l7ysr8,recording:98lc-l7ysr8,record_id_prefix,medium,80,0.000703
1,tong-hop-bahnar_21_11_2025_tl3036_new,recording:yt_49:tong-hop-bahnar_21_11_2025_tl3...,record_id_prefix,medium,80,0.000703
2,luk_01,recording:luk_01,record_id_prefix,medium,80,0.000703
3,pheh4zrlm0i,recording:yt_52:pheh4zrlm0i,record_id_prefix,medium,78,0.000685
4,tong-hop-bahnar_10_12_2025_tl3111_new-1,recording:yt_52:tong-hop-bahnar_10_12_2025_tl3...,record_id_prefix,medium,78,0.000685
5,knnzr8lozj0,recording:knnzr8lozj0,record_id_prefix,medium,74,0.000650
6,vejqiifkj-y,recording:vejqiifkj-y,record_id_prefix,medium,73,0.000641
7,mrk_14,recording:mrk_14,record_id_prefix,medium,72,0.000633
8,nnyipsdl6b8,recording:nnyipsdl6b8,record_id_prefix,medium,72,0.000633
9,g7au5lfprzm,recording:g7au5lfprzm,record_id_prefix,medium,71,0.000624


Train eligible group_confidence: medium=113,830 low=0 unresolved=0
GROUP_REVIEW_APPROVED: False
Wrote: /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/data/audit/group_source_summary.csv
Wrote: /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/data/audit/train_largest_groups.csv
Wrote: /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/data/audit/train_group_id_review.csv
Wrote: /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/data/audit/validation_group_id_review.csv
Wrote: /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/data/audit/test_group_id_review.csv
Wrote: /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/data/audit/ambiguous_group_suffixes.csv
Wrote: /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-sys

## Step 3b — Waveform QA (decode accessible eval only)

Stream **validation** and **test** parquet shards only. Decode the ~205 + ~10 accessible eligible rows.

**Train waveform QA is deferred** to Notebook 02 data-loader / preflight. This notebook must not download or scan all 113,830 train audio records.

Hard failures (`ok=False` / `hard_ok=False`) exclude a row from frozen test. Quality warnings (`near_silent`, `heavy_clipping`, `duration_mismatch`) are logged but do **not** auto-drop.


In [20]:
# Cell 12b — Decode QA via streaming Parquet (validation + test ONLY)
# Train waveform QA intentionally omitted — deferred to Notebook 02.
# QA join key is always (source_split, record_id).
import io
from pathlib import Path

import soundfile as sf
from datasets import Audio, load_dataset


def _decode_audio_dict(audio):
    """Decode HF audio dict without torchcodec (bytes via soundfile)."""
    if audio is None:
        return None, None
    if not isinstance(audio, dict):
        return None, None
    if audio.get("array") is not None:
        return audio.get("array"), audio.get("sampling_rate")
    raw = audio.get("bytes")
    if raw:
        array, sr = sf.read(io.BytesIO(raw), always_2d=False)
        return array, int(sr)
    path = audio.get("path")
    if path and Path(path).exists():
        array, sr = sf.read(path, always_2d=False)
        return array, int(sr)
    return None, None


def qa_rows_from_parquet_streaming(
    split_name: str,
    record_ids: set[str],
    meta_dur_by_split_id: dict,
) -> pd.DataFrame:
    """Stream only this split's parquet shards; decode audio when id matches."""
    if not record_ids:
        return pd.DataFrame()

    urls = parquet_by_split.get(split_name) or []
    if not urls:
        raise ValueError(f"No parquet URLs for split={split_name}. Re-run Cell 5.")

    needed = set(record_ids)
    target_n = len(needed)
    print(
        f"  {split_name}: streaming {len(urls)} parquet shard(s); "
        f"looking for {target_n:,} record_id(s)"
    )

    ds = load_dataset(
        "parquet",
        data_files=urls,
        split="train",
        streaming=True,
    )
    if "audio" in ds.column_names:
        ds = ds.cast_column("audio", Audio(decode=False))

    rows = []
    scanned = 0
    for ex in ds:
        scanned += 1
        rid = str(ex.get("id") if ex.get("id") is not None else "")
        if rid not in needed:
            if scanned % 1000 == 0:
                print(f"    ... scanned {scanned:,}, matched {len(rows):,}/{target_n:,}")
            continue

        array, sr = _decode_audio_dict(ex.get("audio"))
        expected_dur = meta_dur_by_split_id.get((split_name, rid))
        qa = check_waveform(
            array,
            sr,
            min_duration=AUDIO_MIN_DURATION,
            max_duration=AUDIO_MAX_DURATION,
            expected_duration=expected_dur,
        )
        rows.append({
            "source_split": split_name,
            "record_id": rid,
            "qa_ok": qa["ok"],
            "qa_hard_ok": qa["hard_ok"],
            "qa_reason": qa["reason"],
            "qa_quality_warnings": "|".join(qa.get("quality_warnings") or []),
            "qa_duration_sec": qa["duration_sec"],
            "qa_duration_difference_sec": qa.get("duration_difference_sec"),
            "qa_sampling_rate": qa["sampling_rate"],
            "qa_clip_ratio": qa["clip_ratio"],
            "qa_rms": qa["rms"],
        })
        needed.discard(rid)
        if scanned % 200 == 0 or not needed:
            print(f"    ... scanned {scanned:,}, matched {len(rows):,}/{target_n:,}")
        if not needed:
            break

    if needed:
        print(f"  WARNING: {len(needed)} record_id(s) not found while streaming {split_name}")
    else:
        print(f"  {split_name}: done — matched {len(rows):,} after scanning {scanned:,} rows")
    return pd.DataFrame(rows)


import sys
if "bahnar-s2tt-thesis" not in sys.executable.replace("\\\\", "/") and ".venv" not in sys.executable:
    print("WARNING: kernel is", sys.executable, "— prefer Python (bahnar-s2tt) / thesis .venv")

val_elig = metadata["validation"].loc[metadata["validation"]["eligible_rq1_metadata"]].copy()
test_elig = metadata["test"].loc[metadata["test"]["eligible_rq1_metadata"]].copy()
val_elig["record_id"] = val_elig["record_id"].astype(str)
test_elig["record_id"] = test_elig["record_id"].astype(str)
print(
    f"Frozen-test metadata candidates: "
    f"val={len(val_elig)} test={len(test_elig)} total={len(val_elig)+len(test_elig)}"
)
print("NOTE: Train waveform QA is deferred to Notebook 02 (not run here).")

# Cross-split record_id audit (informational; QA still keyed by split+id)
val_ids = set(val_elig["record_id"])
test_ids = set(test_elig["record_id"])
cross_ids = sorted(val_ids & test_ids)
cross_audit = pd.DataFrame({
    "record_id": cross_ids,
    "in_validation_eligible": True,
    "in_test_eligible": True,
})
cross_audit.to_csv(DIRS["audit"] / "evaluation_cross_split_id_audit.csv", index=False)
print(
    f"Cross-split eligible record_id overlap: {len(cross_ids)} "
    f"(reported only; QA join uses (source_split, record_id))"
)

meta_dur_by_split_id = {}
for split_name, _df in (("validation", val_elig), ("test", test_elig)):
    for rid, dur in zip(_df["record_id"], _df["duration_seconds"]):
        meta_dur_by_split_id[(split_name, str(rid))] = float(dur) if pd.notna(dur) else None

print("QA validation (streaming parquet)...")
qa_val = qa_rows_from_parquet_streaming(
    "validation", set(val_elig["record_id"]), meta_dur_by_split_id
)
print("QA test (streaming parquet)...")
qa_test = qa_rows_from_parquet_streaming(
    "test", set(test_elig["record_id"]), meta_dur_by_split_id
)
qa_eval = pd.concat([qa_val, qa_test], ignore_index=True)

if len(qa_eval):
    qa_eval["record_id"] = qa_eval["record_id"].astype(str)
    dup_mask = qa_eval.duplicated(subset=["source_split", "record_id"], keep=False)
    if dup_mask.any():
        dup_path = DIRS["audit"] / "audio_qa_duplicate_composite_keys.csv"
        qa_eval.loc[dup_mask].to_csv(dup_path, index=False)
        raise RuntimeError(
            f"Duplicate QA composite keys (source_split, record_id) found. "
            f"See {dup_path}. Blocking further steps."
        )

qa_eval.to_csv(DIRS["audit"] / "audio_qa_eval_candidates.csv", index=False)

qa_eval_summary = (
    qa_eval.groupby(["source_split", "qa_ok", "qa_hard_ok", "qa_reason"], dropna=False)
    .size()
    .reset_index(name="rows")
) if len(qa_eval) else pd.DataFrame()
display(qa_eval_summary)
print(
    "Eval QA hours (hard_ok only):",
    round(float(qa_eval.loc[qa_eval["qa_hard_ok"], "qa_duration_sec"].sum()) / 3600, 3)
    if len(qa_eval) and qa_eval["qa_hard_ok"].any()
    else 0,
)
warn_n = int((qa_eval["qa_quality_warnings"].fillna("") != "").sum()) if len(qa_eval) else 0
print(f"Eval rows with quality warnings (kept if hard_ok): {warn_n}")

QA_JOIN_COLS = [
    "qa_ok", "qa_hard_ok", "qa_reason", "qa_quality_warnings",
    "qa_duration_sec", "qa_duration_difference_sec",
]
for split in ("validation", "test"):
    df = metadata[split].copy()
    df["record_id"] = df["record_id"].astype(str)
    for c in QA_JOIN_COLS:
        if c in df.columns:
            df = df.drop(columns=[c])
    split_qa = (
        qa_eval.loc[qa_eval["source_split"] == split, ["record_id", *QA_JOIN_COLS]]
        .drop_duplicates(subset=["record_id"], keep="first")
        if len(qa_eval)
        else pd.DataFrame(columns=["record_id", *QA_JOIN_COLS])
    )
    metadata[split] = df.merge(split_qa, on="record_id", how="left")

bad_eval = qa_eval.loc[~qa_eval["qa_hard_ok"]].copy() if len(qa_eval) else pd.DataFrame()
bad_eval.to_csv(DIRS["audit"] / "audio_qa_eval_excluded.csv", index=False)
print(f"Eval audio excluded after hard QA failures: {len(bad_eval)}")
print("Wrote:", DIRS["audit"] / "evaluation_cross_split_id_audit.csv")


Frozen-test metadata candidates: val=205 test=10 total=215
NOTE: Train waveform QA is deferred to Notebook 02 (not run here).
Cross-split eligible record_id overlap: 0 (reported only; QA join uses (source_split, record_id))
QA validation (streaming parquet)...
  validation: streaming 1 parquet shard(s); looking for 205 record_id(s)
    ... scanned 1,000, matched 10/205
    ... scanned 2,000, matched 10/205
    ... scanned 2,600, matched 169/205
    ... scanned 2,636, matched 205/205
  validation: done — matched 205 after scanning 2,636 rows
QA test (streaming parquet)...
  test: streaming 1 parquet shard(s); looking for 10 record_id(s)
    ... scanned 10, matched 10/10
  test: done — matched 10 after scanning 10 rows


,source_split,qa_ok,qa_hard_ok,qa_reason,rows
0,test,True,True,ok,10
1,validation,True,True,ok,205


Eval QA hours (hard_ok only): 0.692
Eval rows with quality warnings (kept if hard_ok): 0
Eval audio excluded after hard QA failures: 0
Wrote: /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/data/audit/evaluation_cross_split_id_audit.csv


## Step 4 — Freeze accessible evaluation set (`rq1_test`)

Build frozen test from **validation+test rows that are metadata-eligible and pass waveform QA**.

Rules after freeze:

- Do not use for checkpoint selection
- Do not tune hyperparameters on it
- Do not put these rows into train
- Ignore `audio=None` rows for now


In [21]:
# Cell 13 — Frozen rq1_test candidate (hard QA pass only)
def is_qa_hard_ok(df):
    if "qa_hard_ok" in df.columns:
        return df["eligible_rq1_metadata"] & df["qa_hard_ok"].fillna(False)
    if "qa_ok" in df.columns:
        return df["eligible_rq1_metadata"] & df["qa_ok"].fillna(False)
    return df["eligible_rq1_metadata"] & False

eval_candidate = pd.concat(
    [
        metadata["validation"].loc[is_qa_hard_ok(metadata["validation"])],
        metadata["test"].loc[is_qa_hard_ok(metadata["test"])],
    ],
    ignore_index=True,
).copy()
eval_candidate["split"] = "test"
eval_candidate["label_type"] = "accessible_reference_candidate"
eval_candidate["record_id"] = eval_candidate["record_id"].astype(str)

# Composite-key and record_uid uniqueness (block export if violated)
comp_dup = eval_candidate.duplicated(subset=["source_split", "record_id"], keep=False)
uid_dup = (
    eval_candidate.duplicated(subset=["record_uid"], keep=False)
    if "record_uid" in eval_candidate.columns
    else pd.Series(False, index=eval_candidate.index)
)
EVAL_IDENTITY_OK = True
if comp_dup.any():
    EVAL_IDENTITY_OK = False
    path = DIRS["audit"] / "evaluation_duplicate_composite_keys.csv"
    eval_candidate.loc[comp_dup].to_csv(path, index=False)
    print(f"ERROR: duplicate (source_split, record_id) in frozen test — {path}")
if "record_uid" in eval_candidate.columns and uid_dup.any():
    EVAL_IDENTITY_OK = False
    path = DIRS["audit"] / "evaluation_duplicate_record_uid.csv"
    eval_candidate.loc[uid_dup].to_csv(path, index=False)
    print(f"ERROR: duplicate record_uid in frozen test — {path}")
if not EVAL_IDENTITY_OK:
    raise RuntimeError(
        "Frozen-test identity checks failed (composite key and/or record_uid). "
        "Final manifest export must be blocked."
    )

eval_duplicate_report = pd.DataFrame([{
    "rows": len(eval_candidate),
    "unique_record_id": int(eval_candidate["record_id"].nunique()),
    "unique_composite_split_record": int(
        eval_candidate[["source_split", "record_id"]].drop_duplicates().shape[0]
    ),
    "unique_record_uid": int(eval_candidate["record_uid"].nunique()) if "record_uid" in eval_candidate.columns else None,
    "unique_audio_key": int(eval_candidate["audio_key"].nunique()) if "audio_key" in eval_candidate.columns else None,
    "unique_pair_key": int(eval_candidate["pair_key"].nunique()) if "pair_key" in eval_candidate.columns else None,
    "unique_source_labels": int(eval_candidate["source_label"].nunique(dropna=True)),
    "unique_recording_groups": int(eval_candidate["recording_group_id"].nunique(dropna=True)),
    "hours_meta": round(float(eval_candidate["duration_seconds"].fillna(0).sum()) / 3600, 4),
    "identity_ok": True,
}])
display(eval_duplicate_report)
eval_duplicate_report.to_csv(DIRS["audit"] / "evaluation_candidate_duplicate_report.csv", index=False)

if DATASET_REVISION_AFTER == EXPECTED_DATASET_REVISION:
    if len(eval_candidate) != EXPECTED_FROZEN_TEST_ROWS:
        print(
            f"WARNING: frozen-test hard-pass rows={len(eval_candidate)} "
            f"(expected {EXPECTED_FROZEN_TEST_ROWS} when revision locked). "
            "Inspect audio_qa_eval_excluded.csv. Final export will fail integrity if still wrong."
        )
else:
    print("Revision differs from expected — frozen-test size check relaxed to warning only.")

print(f"Frozen-test candidate rows (hard_ok): {len(eval_candidate)}")


/var/folders/26/mcdkmc514230s9n69w04jxm40000gn/T/ipykernel_49831/1893828260.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df["eligible_rq1_metadata"] & df["qa_hard_ok"].fillna(False)


,rows,unique_record_id,unique_composite_split_record,unique_record_uid,unique_audio_key,unique_pair_key,unique_source_labels,unique_recording_groups,hours_meta,identity_ok
0,215,215,215,215,215,209,6,5,0.6919,True


Frozen-test candidate rows (hard_ok): 215


In [22]:
# Cell 14 — Train eligibility, exact duplicates and leakage flags
train_pool = metadata["train"].loc[
    metadata["train"]["eligible_rq1_metadata"]
].copy()
train_pool["label_type"] = "pseudo_label"

exact_subset = ["audio_key", "pair_key"]
exact_duplicate_mask = train_pool.duplicated(subset=exact_subset, keep="first")
exact_duplicates = train_pool.loc[exact_duplicate_mask].copy()

if DROP_EXACT_DUPLICATE_ROWS:
    train_pool = train_pool.loc[~exact_duplicate_mask].copy()

eval_ids = set(eval_candidate.loc[has_text(eval_candidate["record_id"]), "record_id"].astype(str))
eval_audio_keys = set(eval_candidate.loc[has_text(eval_candidate["audio_key"]), "audio_key"])
eval_pair_keys = set(eval_candidate["pair_key"])
eval_groups = set(eval_candidate["recording_group_id"].astype(str))

train_pool["leak_record_id"] = train_pool["record_id"].astype(str).isin(eval_ids)
train_pool["leak_audio_key"] = train_pool["audio_key"].isin(eval_audio_keys) & has_text(train_pool["audio_key"])
train_pool["leak_text_pair"] = train_pool["pair_key"].isin(eval_pair_keys)
train_pool["leak_recording_group"] = train_pool["recording_group_id"].astype(str).isin(eval_groups)

hard_leakage = (
    train_pool["leak_record_id"]
    | train_pool["leak_audio_key"]
    | train_pool["leak_recording_group"]
)
leakage_excluded = train_pool.loc[
    hard_leakage | (train_pool["leak_text_pair"] if REMOVE_EXACT_TEXT_PAIR_OVERLAP else False)
].copy()

keep_mask = ~hard_leakage
if REMOVE_EXACT_TEXT_PAIR_OVERLAP:
    keep_mask &= ~train_pool["leak_text_pair"]
clean_train_pool = train_pool.loc[keep_mask].copy()

leakage_summary = pd.DataFrame([{
    "eligible_train_before_dedup": int(metadata["train"]["eligible_rq1_metadata"].sum()),
    "exact_duplicates_removed": len(exact_duplicates),
    "record_id_overlap_rows": int(train_pool["leak_record_id"].sum()),
    "audio_key_overlap_rows": int(train_pool["leak_audio_key"].sum()),
    "recording_group_overlap_rows": int(train_pool["leak_recording_group"].sum()),
    "text_pair_overlap_rows": int(train_pool["leak_text_pair"].sum()),
    "total_leakage_excluded": len(leakage_excluded),
    "clean_train_pool": len(clean_train_pool),
    "clean_train_groups": int(clean_train_pool["recording_group_id"].nunique()),
    "clean_train_hours_meta": round(float(clean_train_pool["duration_seconds"].fillna(0).sum()) / 3600, 3),
}])
display(leakage_summary)

exact_duplicates.to_csv(DIRS["audit"] / "train_exact_duplicates_removed.csv", index=False)
leakage_excluded.to_csv(DIRS["audit"] / "train_rows_excluded_for_test_leakage.csv", index=False)
leakage_summary.to_csv(DIRS["audit"] / "leakage_summary.csv", index=False)


,eligible_train_before_dedup,exact_duplicates_removed,record_id_overlap_rows,audio_key_overlap_rows,recording_group_overlap_rows,text_pair_overlap_rows,total_leakage_excluded,clean_train_pool,clean_train_groups,clean_train_hours_meta
0,113830,0,0,0,0,0,0,113830,4502,654.679


## Step 5 — Group-aware train/validation split

Uses `select_best_group_split` (≥200 deterministic candidates):

1. Minimize |val% − 10%|
2. Soft-balance `source_label` vs full clean train pool
3. Soft-penalty if a multi-group source is missing from validation

**Hard requirements:** zero `group_id` overlap; validation row % in **8–12%** or raise (no `split_ready`).

First Phase B run stops here if `GROUP_REVIEW_APPROVED=False`. After reviewing largest groups:

1. Set `GROUP_REVIEW_APPROVED=True` in Cell 3
2. Rerun from Cell 3 (do not auto-flip this flag)


In [23]:
# Cell 15 — Execute controlled group split only after review approval
split_ready = False
rq1_train = None
rq1_validation = None
split_selection = None

if not GROUP_REVIEW_APPROVED:
    print("STOPPED AT GROUP REVIEW GATE")
    print("Review these files before approving:")
    print(" -", DIRS["audit"] / "group_source_summary.csv")
    print(" -", DIRS["audit"] / "train_largest_groups.csv")
    print(" -", DIRS["audit"] / "train_group_id_review.csv")
    print(" -", DIRS["audit"] / "validation_group_id_review.csv")
    print(" -", DIRS["audit"] / "test_group_id_review.csv")
    print(" -", DIRS["audit"] / "ambiguous_group_suffixes.csv")
    print(" -", DIRS["audit"] / "group_id_cross_source_collisions.csv")
    print(" -", DIRS["audit"] / "audio_qa_eval_candidates.csv")
    print("Then set GROUP_REVIEW_APPROVED=True in Cell 3 and rerun from Cell 3.")
else:
    if clean_train_pool["group_id"].nunique() < 2:
        raise ValueError("At least two distinct group_id values are required for a group split")

    split_selection = select_best_group_split(
        clean_train_pool,
        group_col="group_id",
        source_col="source_label",
        test_size=VALIDATION_FRACTION,
        min_val_fraction=VALIDATION_FRACTION_MIN,
        max_val_fraction=VALIDATION_FRACTION_MAX,
        seed=SEED,
        n_candidates=SPLIT_N_CANDIDATES,
    )

    rq1_train = clean_train_pool.iloc[split_selection["train_idx"]].copy()
    rq1_validation = clean_train_pool.iloc[split_selection["val_idx"]].copy()
    rq1_train["split"] = "train"
    rq1_validation["split"] = "validation"

    group_overlap = set(rq1_train["group_id"]) & set(rq1_validation["group_id"])
    if group_overlap:
        raise RuntimeError(f"Group leakage detected: {list(group_overlap)[:5]}")
    if set(rq1_train["pair_key"]) & set(eval_candidate["pair_key"]):
        raise RuntimeError("train/test pair_key overlap")
    if set(rq1_validation["pair_key"]) & set(eval_candidate["pair_key"]):
        raise RuntimeError("validation/test pair_key overlap")
    if set(rq1_train["record_id"].astype(str)) & set(eval_candidate["record_id"].astype(str)):
        raise RuntimeError("train/test record_id overlap")
    if set(rq1_validation["record_id"].astype(str)) & set(eval_candidate["record_id"].astype(str)):
        raise RuntimeError("validation/test record_id overlap")

    val_pct = 100.0 * len(rq1_validation) / max(len(clean_train_pool), 1)
    if not (100 * VALIDATION_FRACTION_MIN <= val_pct <= 100 * VALIDATION_FRACTION_MAX):
        raise RuntimeError(
            f"Validation {val_pct:.2f}% outside 8–12% band — refusing split_ready"
        )

    split_ready = True

    train_src = rq1_train["source_label"].fillna("__NA__").value_counts(normalize=True).rename("train")
    val_src = rq1_validation["source_label"].fillna("__NA__").value_counts(normalize=True).rename("validation")
    src_dist = pd.concat([train_src, val_src], axis=1).fillna(0.0)
    src_dist.to_csv(DIRS["audit"] / "train_validation_source_label_distribution.csv")

    print(f"Train rows:        {len(rq1_train):,}")
    print(f"Validation rows:   {len(rq1_validation):,}")
    print(f"Validation %:      {val_pct:.2f}%")
    print(f"Train groups:      {rq1_train['group_id'].nunique():,}")
    print(f"Validation groups: {rq1_validation['group_id'].nunique():,}")
    print(f"Train hours(meta): {rq1_train['duration_seconds'].fillna(0).sum()/3600:.2f}")
    print(f"Val hours(meta):   {rq1_validation['duration_seconds'].fillna(0).sum()/3600:.2f}")
    print(f"Selected candidate seed: {split_selection['candidate_seed']}")
    print(f"Selected score: {split_selection['score']:.6f}")
    print(f"size_error: {split_selection['size_error']:.6f}")
    print(f"source_balance_l1: {split_selection['source_balance_l1']:.6f}")
    print("Warnings:", split_selection.get("warnings"))
    print("Group overlap: 0")
    display(src_dist.head(30))


STOPPED AT GROUP REVIEW GATE
Review these files before approving:
 - /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/data/audit/group_source_summary.csv
 - /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/data/audit/train_largest_groups.csv
 - /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/data/audit/train_group_id_review.csv
 - /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/data/audit/validation_group_id_review.csv
 - /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/data/audit/test_group_id_review.csv
 - /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/data/audit/ambiguous_group_suffixes.csv
 - /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/data/audit/group_id_cross_source_co

## Step 6 — Export immutable manifests

Notebook 01 is complete only when these exist:

```text
data/manifests/rq1_train.csv|.parquet
data/manifests/rq1_validation.csv|.parquet
data/manifests/rq1_test.csv|.parquet
data/manifests/split_summary.json
```


In [24]:
# Cell 16 — Manifest columns and exporters (only if integrity pre-checks pass)
MANIFEST_COLUMNS = [
    "record_uid", "record_id", "source_split", "parquet_file", "shard_row_index",
    "audio_path", "audio_available", "duration_seconds",
    "source_label", "speaker_id",
    "recording_group_id", "group_id", "group_source", "group_confidence",
    "text_bahnar", "text_vi", "text_en", "pair_key", "split", "label_type",
    "qa_ok", "qa_hard_ok", "qa_reason", "qa_quality_warnings",
    "qa_duration_sec", "qa_duration_difference_sec",
]

def manifest_view(df):
    available = [c for c in MANIFEST_COLUMNS if c in df.columns]
    return df[available].copy().sort_values(["group_id", "record_uid"]).reset_index(drop=True)

def write_manifest(df, stem):
    csv_path = DIRS["manifests"] / f"{stem}.csv"
    parquet_path = DIRS["manifests"] / f"{stem}.parquet"
    view = manifest_view(df)
    view.to_csv(csv_path, index=False)
    view.to_parquet(parquet_path, index=False)
    return csv_path, parquet_path

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def preflight_export_checks():
    checks = {}
    checks["split_ready"] = bool(split_ready)
    checks["train_non_empty"] = split_ready and len(rq1_train) > 0
    checks["validation_non_empty"] = split_ready and len(rq1_validation) > 0
    checks["test_non_empty"] = len(eval_candidate) > 0
    if split_ready:
        checks["no_train_validation_group_overlap"] = not bool(
            set(rq1_train["group_id"]) & set(rq1_validation["group_id"])
        )
        rid_t = set(rq1_train["record_id"].astype(str))
        rid_v = set(rq1_validation["record_id"].astype(str))
        rid_e = set(eval_candidate["record_id"].astype(str))
        checks["no_record_id_overlap_tvt"] = (
            not (rid_t & rid_v) and not (rid_t & rid_e) and not (rid_v & rid_e)
        )
        if "audio_key" in rq1_train.columns:
            checks["no_audio_key_overlap_tvt"] = not bool(
                set(rq1_train["audio_key"]) & set(rq1_validation["audio_key"])
            ) and not bool(
                set(rq1_train["audio_key"]) & set(eval_candidate["audio_key"])
            ) and not bool(
                set(rq1_validation["audio_key"]) & set(eval_candidate["audio_key"])
            )
        checks["no_pair_key_overlap_with_test"] = not bool(
            set(rq1_train["pair_key"]) & set(eval_candidate["pair_key"])
        ) and not bool(
            set(rq1_validation["pair_key"]) & set(eval_candidate["pair_key"])
        )
        val_frac = len(rq1_validation) / max(len(clean_train_pool), 1)
        checks["validation_fraction_in_8_12"] = (
            VALIDATION_FRACTION_MIN <= val_frac <= VALIDATION_FRACTION_MAX
        )
    if DATASET_REVISION_AFTER == EXPECTED_DATASET_REVISION:
        checks["frozen_test_rows_215"] = len(eval_candidate) == EXPECTED_FROZEN_TEST_ROWS
    checks["eval_identity_ok"] = bool(globals().get("EVAL_IDENTITY_OK", True))
    if "record_uid" in eval_candidate.columns:
        checks["frozen_test_record_uid_unique"] = (
            int(eval_candidate["record_uid"].nunique()) == len(eval_candidate)
        )
    checks["frozen_test_composite_key_unique"] = (
        int(eval_candidate[["source_split", "record_id"]].drop_duplicates().shape[0])
        == len(eval_candidate)
    )
    checks["dataset_sha_locked"] = (
        DATASET_REVISION_BEFORE == EXPECTED_DATASET_REVISION
        and DATASET_REVISION_AFTER == EXPECTED_DATASET_REVISION
    )
    # All frozen-test audio hard-decoded, or hard failures explicitly logged
    excluded_path = DIRS["audit"] / "audio_qa_eval_excluded.csv"
    qa_path = DIRS["audit"] / "audio_qa_eval_candidates.csv"
    checks["audio_qa_report_exists"] = qa_path.exists()
    if qa_path.exists():
        qa_df = pd.read_csv(qa_path)
        hard_ok_n = int(qa_df["qa_hard_ok"].fillna(False).astype(bool).sum()) if "qa_hard_ok" in qa_df.columns else 0
        checks["frozen_test_matches_hard_ok_qa"] = hard_ok_n == len(eval_candidate)
    return checks

draft_test_csv, draft_test_parquet = write_manifest(eval_candidate, "rq1_test_candidate")
clean_train_pool.to_parquet(DIRS["audit"] / "clean_train_pool_before_split.parquet", index=False)

preflight = preflight_export_checks()
preflight_df = pd.DataFrame([{"check": k, "passed": bool(v)} for k, v in preflight.items()])
display(preflight_df)
preflight_df.to_csv(DIRS["audit"] / "export_preflight_checks.csv", index=False)

if split_ready and preflight_df["passed"].all():
    train_csv, train_parquet = write_manifest(rq1_train, "rq1_train")
    validation_csv, validation_parquet = write_manifest(rq1_validation, "rq1_validation")
    test_csv, test_parquet = write_manifest(eval_candidate, "rq1_test")

    summary = {
        "dataset_id": DATASET_ID,
        "config_name": CONFIG_NAME,
        "expected_dataset_revision": EXPECTED_DATASET_REVISION,
        "dataset_commit_sha": DATASET_REVISION_AFTER,
        "dataset_commit_sha_before": DATASET_REVISION_BEFORE,
        "dataset_commit_sha_after": DATASET_REVISION_AFTER,
        "dataset_server_revisions_informational": server_revisions,
        "seed": SEED,
        "validation_fraction_requested": VALIDATION_FRACTION,
        "validation_fraction_band": [VALIDATION_FRACTION_MIN, VALIDATION_FRACTION_MAX],
        "split_n_candidates": SPLIT_N_CANDIDATES,
        "split_selection": {
            "candidate_seed": split_selection["candidate_seed"] if split_selection else None,
            "score": split_selection["score"] if split_selection else None,
            "size_error": split_selection["size_error"] if split_selection else None,
            "source_balance_l1": split_selection["source_balance_l1"] if split_selection else None,
            "val_fraction": split_selection["val_fraction"] if split_selection else None,
            "warnings": split_selection.get("warnings") if split_selection else None,
        },
        "drop_exact_duplicate_rows": DROP_EXACT_DUPLICATE_ROWS,
        "remove_exact_text_pair_overlap": REMOVE_EXACT_TEXT_PAIR_OVERLAP,
        "train_waveform_qa": "deferred_to_notebook_02",
        "frozen_test_rows": len(eval_candidate),
        "counts": {
            "train": len(rq1_train),
            "validation": len(rq1_validation),
            "test": len(eval_candidate),
            "train_groups": int(rq1_train["group_id"].nunique()),
            "validation_groups": int(rq1_validation["group_id"].nunique()),
            "test_groups": int(eval_candidate["group_id"].nunique()),
            "train_hours_meta": round(float(rq1_train["duration_seconds"].fillna(0).sum()) / 3600, 3),
            "validation_hours_meta": round(float(rq1_validation["duration_seconds"].fillna(0).sum()) / 3600, 3),
            "test_hours": round(float(eval_candidate["duration_seconds"].fillna(0).sum()) / 3600, 4),
            "validation_row_pct_of_clean_pool": round(
                100.0 * len(rq1_validation) / max(len(clean_train_pool), 1), 4
            ),
        },
        "manifest_sha256": {
            "rq1_train.csv": sha256_file(train_csv),
            "rq1_validation.csv": sha256_file(validation_csv),
            "rq1_test.csv": sha256_file(test_csv),
        },
    }
    with open(DIRS["manifests"] / "split_summary.json", "w", encoding="utf-8") as handle:
        json.dump(summary, handle, ensure_ascii=False, indent=2)

    print("FINAL MANIFESTS SAVED")
    for path in (train_csv, train_parquet, validation_csv, validation_parquet, test_csv, test_parquet):
        print(" -", path)
    print(" -", DIRS["manifests"] / "split_summary.json")
elif split_ready:
    raise RuntimeError(
        "Export blocked: one or more mandatory preflight checks failed. "
        "See data/audit/export_preflight_checks.csv — manifests NOT written."
    )
else:
    print("Draft evaluation candidate saved:", draft_test_csv)
    print("Final train/validation/test manifests were NOT written (group review pending or split not ready).")


,check,passed
0,split_ready,False
1,train_non_empty,False
2,validation_non_empty,False
3,test_non_empty,True
4,frozen_test_rows_215,True
5,eval_identity_ok,True
6,frozen_test_record_uid_unique,True
7,frozen_test_composite_key_unique,True
8,dataset_sha_locked,True
9,audio_qa_report_exists,True


Draft evaluation candidate saved: /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/data/manifests/rq1_test_candidate.csv
Final train/validation/test manifests were NOT written (group review pending or split not ready).


In [25]:
# Cell 17 — Integrity checks and final report
if split_ready:
    checks = {
        "train_non_empty": len(rq1_train) > 0,
        "validation_non_empty": len(rq1_validation) > 0,
        "test_non_empty": len(eval_candidate) > 0,
        "no_train_validation_group_overlap": not bool(
            set(rq1_train["group_id"]) & set(rq1_validation["group_id"])
        ),
        "no_train_validation_record_overlap": not bool(
            set(rq1_train["record_id"].astype(str)) & set(rq1_validation["record_id"].astype(str))
        ),
        "no_train_test_record_overlap": not bool(
            set(rq1_train["record_id"].astype(str)) & set(eval_candidate["record_id"].astype(str))
        ),
        "no_validation_test_record_overlap": not bool(
            set(rq1_validation["record_id"].astype(str)) & set(eval_candidate["record_id"].astype(str))
        ),
        "no_train_test_pair_overlap": not bool(
            set(rq1_train["pair_key"]) & set(eval_candidate["pair_key"])
        ),
        "no_validation_test_pair_overlap": not bool(
            set(rq1_validation["pair_key"]) & set(eval_candidate["pair_key"])
        ),
        "validation_fraction_in_8_12": (
            VALIDATION_FRACTION_MIN
            <= (len(rq1_validation) / max(len(clean_train_pool), 1))
            <= VALIDATION_FRACTION_MAX
        ),
        "dataset_sha_in_summary": False,
        "manifests_exist": all((
            (DIRS["manifests"] / "rq1_train.parquet").exists(),
            (DIRS["manifests"] / "rq1_validation.parquet").exists(),
            (DIRS["manifests"] / "rq1_test.parquet").exists(),
            (DIRS["manifests"] / "split_summary.json").exists(),
        )),
        "manifest_sha256_present": False,
    }
    if "audio_key" in rq1_train.columns:
        checks["no_train_validation_audio_key_overlap"] = not bool(
            set(rq1_train["audio_key"]) & set(rq1_validation["audio_key"])
        )
        checks["no_train_test_audio_key_overlap"] = not bool(
            set(rq1_train["audio_key"]) & set(eval_candidate["audio_key"])
        )
        checks["no_validation_test_audio_key_overlap"] = not bool(
            set(rq1_validation["audio_key"]) & set(eval_candidate["audio_key"])
        )
    if DATASET_REVISION_AFTER == EXPECTED_DATASET_REVISION:
        checks["frozen_test_rows_215"] = len(eval_candidate) == EXPECTED_FROZEN_TEST_ROWS

    summary_path = DIRS["manifests"] / "split_summary.json"
    if summary_path.exists():
        summary_obj = json.loads(summary_path.read_text(encoding="utf-8"))
        checks["dataset_sha_in_summary"] = bool(summary_obj.get("dataset_commit_sha"))
        checks["manifest_sha256_present"] = bool(summary_obj.get("manifest_sha256"))

    checks_df = pd.DataFrame(
        [{"check": key, "passed": bool(value)} for key, value in checks.items()]
    )
    display(checks_df)
    checks_df.to_csv(DIRS["audit"] / "final_integrity_checks.csv", index=False)
    if not checks_df["passed"].all():
        raise RuntimeError("At least one final integrity check failed — see final_integrity_checks.csv")

    print("Notebook 01 Phase B completed successfully.")
    print("Next notebook: 02_train_asr.ipynb (includes deferred train waveform QA / preflight).")
else:
    print("Notebook 01 Phase B — review pending.")
    print("Next action: inspect group + audio QA CSVs, then set GROUP_REVIEW_APPROVED=True.")
    print("GROUP_REVIEW_APPROVED remains", GROUP_REVIEW_APPROVED)


Notebook 01 Phase B — review pending.
Next action: inspect group + audio QA CSVs, then set GROUP_REVIEW_APPROVED=True.
GROUP_REVIEW_APPROVED remains False


# Expected hand-off

Before setting `GROUP_REVIEW_APPROVED=True`, review:

- `data/audit/group_source_summary.csv`
- `data/audit/train_largest_groups.csv`
- `data/audit/*_group_id_review.csv` (stratified)
- `data/audit/ambiguous_group_suffixes.csv`
- `data/audit/group_id_cross_source_collisions.csv`
- `data/audit/audio_qa_eval_candidates.csv`

Final artifacts (only after approval + integrity pass):

```
data/manifests/rq1_train.csv
data/manifests/rq1_train.parquet
data/manifests/rq1_validation.csv
data/manifests/rq1_validation.parquet
data/manifests/rq1_test.csv
data/manifests/rq1_test.parquet
data/manifests/split_summary.json
```

Train waveform QA is **not** part of Notebook 01 — deferred to Notebook 02.
